# 모듈 1: SMOTE Ablation 및 Optuna 튜닝

이 노트북은 IDE에 연결된 원격 Colab(또는 GPU) 환경에서 직접 실행할 수 있도록 구성되었습니다.
실행 전 `requirements-colab.txt` 환경이 구성되어 있는지 확인하세요.

> **참고**: 각 셀은 내부 메모리 해제 및 로깅 안전성을 위해 외부 스크립트를 호출(`!python`)하는 방식으로 구성되었습니다.

In [1]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[0]
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print(f"현재 작업 디렉토리: {os.getcwd()}")

현재 작업 디렉토리: /


## 1. 방어 논리용: 임베딩 SMOTE Ablation

제안서에 명시된 "SMOTE"를 임베딩 공간에서 그대로 적용해 봅니다.
End-to-End Fine-Tuning 모델에 비해 구조적 한계(Frozen Encoder + MLP Head)로 인해 성능이 하락한다는 점을 실증합니다.

이 결과는 "왜 제안서의 SMOTE를 쓰지 않고 LLM 증강을 썼는가"에 대한 완벽한 방어 논리가 됩니다.

In [2]:
!python scripts/train_embedding_smote.py

python3: can't open file '//scripts/train_embedding_smote.py': [Errno 2] No such file or directory


## 2. 성능 극대화: Optuna 하이퍼파라미터 튜닝

미리 생성해둔 경고(2) 템플릿 합성 데이터를 동적으로 훈련셋에 주입하면서 최적의 하이퍼파라미터를 찾습니다.

- **목적함수**: Macro-F1 최대화
- **제약 조건**: 긴급(3) Recall >= 0.80 하한선 (미달 시 엄청난 페널티 부여)
- **탐색 변수**: `lr`, `warmup_ratio`, `weight_decay`, `alpha` 가중치, 그리고 **경고(2) 증강 배율(`warning_augment_ratio`)**
- **탐색 횟수**: 30 Trials (TPE 알고리즘 특성상 30회면 충분, MedianPruner로 나쁜 Trial은 1 Epoch 만에 조기 종료)

In [3]:
# GPU 여력이 허락하는 선에서 --n-trials 조절 가능 (A100: 30 ~ 50, T4: 20 ~ 30 권장)
!python scripts/optuna_search.py --n-trials 30

python3: can't open file '//scripts/optuna_search.py': [Errno 2] No such file or directory


## 3. 베스트 파라미터로 최종 학습

위 Optuna 검색에서 출력된 `Best Params`를 `configs/module1_kcelectra_optuna_best.yaml` 파일로 복사하여 작성한 뒤, 아래 셀을 실행하여 최종 모델을 저장합니다.

In [4]:
import yaml

# 예시: Optuna 결과를 여기에 복사해서 사용하세요
best_config = {
    "model": {
        "name": "beomi/KcELECTRA-base-v2022",
        "num_labels": 4,
        "max_length": 128,
        "checkpoint_dir": "models/checkpoints/module1_optuna_best"
    },
    "training": {
        "seed": 42,
        "batch_size": 32,
        "eval_batch_size": 64,
        "num_epochs": 2,
        "lr": 3e-5,  # Optuna 베스트 값으로 변경
        "weight_decay": 0.01,  # Optuna 베스트 값으로 변경
        "warmup_ratio": 0.1,  # Optuna 베스트 값으로 변경
        "grad_accum_steps": 1,
        "fp16": True
    },
    "loss": {
        "type": "ce",  # Focal Loss 대신 가중치 교차 엔트로피 사용 (gamma=0)
        "focal_gamma": 0.0,
        "alpha": [1.0, 1.5, 2.0, 5.0]  # Optuna 베스트 배열(alpha_0~alpha_3)로 변경
    },
    "paths": {
        "checkpoint_dir": "models/checkpoints/module1_optuna_best"
    }
}

with open("configs/module1_kcelectra_optuna_best.yaml", "w") as f:
    yaml.dump(best_config, f)

print("최종 Config 파일 생성 완료!")

FileNotFoundError: [Errno 2] No such file or directory: 'configs/module1_kcelectra_optuna_best.yaml'

In [ ]:
# 생성한 베스트 Config로 본 학습 진행
# warning_augment_ratio 는 Optuna에서 찾은 값을 아래 오버라이드 딕셔너리로 넘겨주세요.

from src.training.trainer import train_module1

result = train_module1(
    config_path="configs/module1_kcelectra_optuna_best.yaml",
    project_root=".",
    override_params={"warning_augment_ratio": 2.0} # Optuna에서 찾은 베스트 비율로 변경
)

print("최종 학습 완료!")